# **Estudo Independente 03** - Coleta de Dados utilizando APIs ou Web Scraping

14-11-2026

**Fonte:** [Dados Abertos - POA](https://dadosabertos.poa.br/), conjunto **Postos de Confiança** (Secretaria Municipal de Planejamento e Gestão). Licença: Creative Commons Atribuição.

**Link dos dados:** `https://dadosabertos.poa.br/dataset/bc544f64-5e86-4a60-8dcb-527c06a24021/resource/bd7dcbce-7800-43bc-a80c-b4a2c012ac76/download/postos_confianca.csv`

## O que são "postos de confiança"?
São as vagas da Prefeitura ocupadas por indicação, e não por concurso: **cargos em comissão (CC)**, **funções gratificadas (FG)** e os cargos pagos por **subsídio** (prefeito, vice e secretários). Cada linha do conjunto de dados é **uma vaga**, e não uma pessoa: o arquivo não traz nomes de servidores.

## Estrutura dos dados
| Campo | Descrição |
|---|---|
| data_extracao | Data em que o retrato dos dados foi extraído do sistema |
| secretaria | Sigla do órgão onde a vaga existe (SMS = Saúde, SMED = Educação, SMF = Fazenda, PGM = Procuradoria-Geral do Município, GP = Gabinete do Prefeito...) |
| descricao | Nome do cargo ou função |
| numero_vaga | Identificador da vaga |
| nivel | Nível: FG1 a FG8, CC13 a CC19 ou SUBSIDIO (quanto maior o número, maior o nível) |
| lei | Lei que criou o posto |
| dtini / dtfim | Início e fim da vigência da vaga (fim vazio = vaga vigente) |

## Como os dados são obtidos
O portal usa CKAN, que oferece uma API pública. O código consulta `package_show`, localiza o link do CSV, baixa os dados e organiza.

## 1. Consulta à API e download do CSV

In [9]:
import requests
import pandas as pd
import matplotlib.pyplot as plt

# ---------- CONSULTA À API DO PORTAL (CKAN) ----------
# A API devolve os metadados do conjunto: título, licença e lista de arquivos
API = "https://dadosabertos.poa.br/api/3/action/package_show"
resp = requests.get(API, params={"id": "postos-de-confianca"}, timeout=30)
resp.raise_for_status()              # interrompe se o servidor responder erro
info = resp.json()["result"]
print("Conjunto:", info["title"])

# ---------- LOCALIZA E BAIXA O CSV ----------
# "resources" lista os arquivos (CSV de dados e PDF do dicionário); pegamos o CSV
url_csv = next(r["url"] for r in info["resources"] if r["format"].upper() == "CSV")
print("Arquivo CSV:", url_csv)

df = pd.read_csv(url_csv, sep=";")    # o arquivo usa ponto e vírgula como separador
print("Registros brutos:", df.shape)
df.head()

Conjunto: Postos de Confiança
Arquivo CSV: https://dadosabertos.poa.br/dataset/bc544f64-5e86-4a60-8dcb-527c06a24021/resource/bd7dcbce-7800-43bc-a80c-b4a2c012ac76/download/postos_confianca.csv
Registros brutos: (2853, 8)


,data_extracao,secretaria,descricao,numero_vaga,nivel,lei,dtini,dtfim
0,2026-10-01 00:00:00,SMC,SECRETARIO MUNICIPAL,1000980,SUBSIDIO,LE,2005-07-01 00:00:00,NaN
1,2026-10-01 00:00:00,SMSEG,SECRETARIO MUNICIPAL,1001905,SUBSIDIO,LE,2005-07-01 00:00:00,NaN
2,2026-10-01 00:00:00,SMS,SECRETARIO MUNICIPAL,1001172,SUBSIDIO,LE,2005-07-01 00:00:00,NaN
3,2026-10-01 00:00:00,GP,PREFEITO,1000001,SUBSIDIO,LOM,2005-07-01 00:00:00,NaN
4,2026-10-01 00:00:00,GP,VICE-PREFEITO,1000002,SUBSIDIO,LOM,2005-07-01 00:00:00,NaN


## 2. Limpeza e estruturação

In [10]:
# ---------- DATAS ----------
# Converte texto em data; valores vazios em dtfim viram NaT
for col in ["data_extracao", "dtini", "dtfim"]:
    df[col] = pd.to_datetime(df[col], errors="coerce")

# ---------- TEXTO ----------
df["descricao"] = df["descricao"].str.strip().str.title()
df["secretaria"] = df["secretaria"].str.strip()
df = df.drop_duplicates().reset_index(drop=True)             # remove linhas repetidas

# ---------- NOVAS COLUNAS ----------
# Agrupa os níveis em 3 tipos de posto para facilitar a análise
def tipo_posto(nivel):
    nivel = str(nivel)
    if nivel.startswith("FG"):
        return "Função gratificada (FG)"
    if nivel.startswith("CC"):
        return "Cargo em comissão (CC)"
    if nivel == "SUBSIDIO":
        return "Subsídio"
    return "Outro"

df["tipo_posto"] = df["nivel"].apply(tipo_posto)
df["ano_inicio"] = df["dtini"].dt.year        # ano em que a vaga passou a vigorar
df["vigente"] = df["dtfim"].isna()            # True = sem data de fim

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2853 entries, 0 to 2852
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   data_extracao  2853 non-null   datetime64[ns]
 1   secretaria     2853 non-null   object        
 2   descricao      2853 non-null   object        
 3   numero_vaga    2853 non-null   int64         
 4   nivel          2853 non-null   object        
 5   lei            2853 non-null   object        
 6   dtini          2853 non-null   datetime64[ns]
 7   dtfim          1 non-null      datetime64[ns]
 8   tipo_posto     2853 non-null   object        
 9   ano_inicio     2853 non-null   int32         
 10  vigente        2853 non-null   bool          
dtypes: bool(1), datetime64[ns](3), int32(1), int64(1), object(5)
memory usage: 214.7+ KB


## 3. Resumo dos dados coletados

In [11]:
print("Total de vagas:", len(df))
print("Vagas vigentes (sem data de fim):", int(df["vigente"].sum()))
print("Secretarias/órgãos:", df["secretaria"].nunique())
print("Tipos de cargo/função diferentes:", df["descricao"].nunique())
print("\nDados extraídos em:", df["data_extracao"].max().date())
print("\nVagas por tipo de posto:")
print(df["tipo_posto"].value_counts())

Total de vagas: 2853
Vagas vigentes (sem data de fim): 2852
Secretarias/órgãos: 21
Tipos de cargo/função diferentes: 106

Dados extraídos em: 2026-10-01

Vagas por tipo de posto:
tipo_posto
Função gratificada (FG)    2056
Cargo em comissão (CC)      773
Subsídio                     24
Name: count, dtype: int64


## 4. Armazenamento em CSV e JSON

In [12]:
# index=False evita gravar a numeração das linhas como coluna extra
df.to_csv("postos_confianca.csv", index=False, encoding="utf-8")

# orient="records" gera uma lista de objetos (um por vaga); date_format="iso" grava datas AAAA-MM-DD
df.to_json("postos_confianca.json", orient="records", force_ascii=False,
           indent=2, date_format="iso")
print("Arquivos salvos: postos_confianca.csv e postos_confianca.json")

Arquivos salvos: postos_confianca.csv e postos_confianca.json


## 5. Download dos arquivos

In [14]:
from google.colab import files
files.download("postos_confianca.csv")
files.download("postos_confianca.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>